In [ ]:
import random
from datasets import load_dataset, Dataset, IterableDataset

# ====================================================================================
# ✨ 데이터셋: seongchaeae/sage-korean-raw-v1
# ✨ 의미: 한국어 텍스트 데이터를 수집한 원본 형태의 코퍼스 데이터셋입니다.
# ✨ 설명: 이 데이터셋은 다양한 출처에서 가져온 한국어 문장들을 모아 놓은 '재료' 같은 것입니다.
#       우리는 이 데이터를 가지고 텍스트 분석이나 패턴 탐색을 해보는 초보자 실습을 진행할 거예요!
# ====================================================================================

# --- 설정 변수 ---
DATASET_NAME = "seongchaeae/sage-korean-raw-v1"
SAMPLE_COUNT = 100  # 실습을 위해 상위 100개만 사용할 거예요!
SPLIT_NAME = 'train'

print("📚✨ 파이썬 AI 코딩 튜터가 준비한 데이터 분석 시간! ✨📚")
print(f"오늘 탐험할 데이터셋: {DATASET_NAME}")
print("-" * 50)

# --- 1. 데이터 로드 및 스트리밍 테스트 (가장 중요!) ---

dataset = None
try:
    # 🚀 튜터가 알려주는 꿀팁: 데이터가 크기 때문에 'streaming=True'를 사용해서
    # 메모리를 효율적으로 관리하는 것이 좋아요! (데이터가 100만 건이라 엄청 크거든요!)
    print(f"🔍 {SPLIT_NAME} 스플릿을 스트리밍 모드로 로드 시도 중... (느슨하게 테스트해볼게요!)")
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print("🎉 스트리밍 로드 성공! 메모리 친화적이에요!")

except Exception as e:
    # 😢 혹시 스트리밍 로드가 안될 경우를 대비한 만능 방지 코드!
    print(f"⚠️ 스트리밍 로드 실패 감지 ({e}). 대신 소량의 데이터를 일반 모드로 다운로드하여 진행할게요.")
    try:
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME)
    except Exception as e_fallback:
        print(f"🛑 에러 발생: 데이터셋 로드 실패. {e_fallback}")
        exit()

# --- 2. 데이터 샘플 추출 및 반복자 준비 ---

if hasattr(dataset, "take"):
    # .take()가 존재하면: 이건 스트리밍 데이터셋(IterableDataset)일 가능성이 높아요!
    print(f"\n✨ 상위 {SAMPLE_COUNT}개 샘플만 반복자(Iterator)로 준비할게요.")
    # ✨ 핵심 패턴: 전체를 리스트로 만들지 않고, take()를 통해 Iterator로 만듭니다.
    sampled_dataset_iterator = iter(dataset.take(SAMPLE_COUNT))
    
    # 이 경우, 나중에 리스트로 변환하기 위해 먼저 데이터를 하나씩 꺼내야 합니다.
    # 실제 실습을 위해 미리 리스트로 변환하여 사용합니다.
    sampled_dataset_list = [next(sampled_dataset_iterator) for _ in range(SAMPLE_COUNT)]

else:
    # 일반 데이터셋 (Dataset)일 경우
    print(f"\n✨ 상위 {SAMPLE_COUNT}개 샘플을 일반 리스트로 준비할게요.")
    sampled_dataset_list = list(dataset.select(range(SAMPLE_COUNT)))


# --- 3. 초보자를 위한 '창의적' 분석 실습: 텍스트 패턴 탐색기 ---

print("\n🌟✨ [실습 시작] 데이터 속 텍스트 패턴을 찾아보자!")
print("🎨 목표: 상위 샘플들의 텍스트 길이와 언어 분포를 분석해, 데이터의 특성을 이해해봅니다.")

# 분석 결과를 저장할 변수들
text_lengths = []      # 각 텍스트의 길이를 기록할 리스트
language_counts = {}   # 언어별(language) 출현 횟수를 셀 딕셔너리

# 모든 샘플 데이터를 순회하며 분석을 진행합니다.
for i, sample in enumerate(sampled_dataset_list):
    # 📝 우리는 'text' 필드에 담긴 문장들을 분석할 거예요.
    text_content = sample.get('text', '')
    
    if text_content:
        length = len(text_content)
        text_lengths.append(length)
        
        # 🌍 어떤 언어들이 섞여 있는지 카운팅!
        language = sample.get('language', 'Unknown')
        language_counts[language] = language_counts.get(language, 0) + 1

    # 🌟 튜터 코멘트: 너무 많은 데이터일 경우, 너무 느려지니 일단 100개에서 멈출게요!
    if i > 0 and i % 50 == 0:
        print(f"   -> 현재 {i+1}번째 샘플 분석 완료...")

print("-" * 50)

# --- 4. 분석 결과 출력 및 해석 (가장 재미있는 부분!) ---

# 💡 분석 1: 텍스트 길이 통계
avg_length = sum(text_lengths) / len(text_lengths) if text_lengths else 0
min_length = min(text_lengths) if text_lengths else 0
max_length = max(text_lengths) if text_lengths else 0

print("✅ 1. 📊 텍스트 길이 분석 결과:")
print(f"   ➡️ 총 분석 샘플 수: {len(text_lengths)}개")
print(f"   ➡️ 평균 텍스트 길이: 약 {avg_length:.2f} 글자")
print(f"   ➡️ 최소 길이: {min_length} 글자 (아마도 간결한 문장이 많다는 뜻!)")
print(f"   ➡️ 최대 길이: {max_length} 글자 (꽤 길고 복잡한 문장이 발견되었어요!)")
print("\n[해석]: 데이터셋의 문장들은 길이가 상당히 다양해요. 특정 주제에 국한되지 않은 '자연스러운' 텍스트가 많다는 걸 알 수 있죠.")


# 💡 분석 2: 언어 분포 분석
print("✅ 2. 🗺️ 데이터셋에 포함된 주요 언어 분포:")
# 정렬된 형태로 출력하면 보기 편해요!
sorted_languages = sorted(language_counts.items(), key=lambda item: item[1], reverse=True)

for language, count in sorted_languages:
    percentage = (count / len(text_lengths)) * 100
    print(f"   🌎 '{language}': {count}건 ({percentage:.1f}%)")

print("\n✨✨ 코딩 마스터 등극! ✨✨")
print("축하해요! 당신은 데이터셋을 성공적으로 로드하고, 텍스트의 구조적 특징(길이, 언어 분포)을 분석하는 멋진 데이터 탐색가가 되었어요!")
print("다음 단계에서는 이 텍스트를 이용해 '이게 상품 리뷰인가요? 아니면 뉴스 기사인가요?' 같은 분류(Classification)에 도전해볼 수 있습니다!")